In [ ]:
import glob
import json
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
h5_files = sorted(glob.glob('./Trial_*/simulation.h5'))

dfs = []
for i, path in enumerate(h5_files, 1):
    with h5py.File(path, 'r') as f:
        cols = list(f['metrics/simulation_metrics'].attrs['columns'])
        arr  = f['metrics/simulation_metrics'][:]
    df = pd.DataFrame(arr, columns=cols)
    df['Trial'] = i
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

In [ ]:
with open('./config.json') as f:
    config = json.load(f)

In [ ]:
finals   = data.groupby('Trial').last()
initials = data.groupby('Trial').first()

gene_cols  = [c for c in finals.columns if c.startswith('Average ') and c != 'Average Age']
death_cols = [c for c in finals.columns if c.startswith('Deaths from ')]

outcome_cols = ['Population Count', 'Cumulative Deaths'] + death_cols + ['Number of Species']

outcome_stats = finals[outcome_cols].agg(['mean', 'std', 'min', 'max']).T
outcome_stats.columns = ['Mean', 'Std', 'Min', 'Max']

gene_drift = pd.DataFrame({
    'Initial mean': initials[gene_cols].mean(),
    'Final mean':   finals[gene_cols].mean(),
    'Final std':    finals[gene_cols].std(),
})
gene_drift['Change'] = gene_drift['Final mean'] - gene_drift['Initial mean']

print('=== Outcome statistics (final state per trial) ===')
display(outcome_stats.round(2))
print('=== Gene drift across trials ===')
display(gene_drift.round(2))

In [ ]:
plt.figure(figsize=(12, 6))
timesteps = data['Timestep'].unique()
# Average Age is a runtime property, not a gene — shown dashed for reference
if config['enable_aging']:
    mean = data.groupby('Timestep')['Average Age'].mean()
    std  = data.groupby('Timestep')['Average Age'].std()
    plt.errorbar(timesteps, mean, yerr=std, label='Average Age', fmt='-o', linestyle='--', alpha=0.6)
# gene_cols is auto-discovered from HDF5 — always in sync with the GENES registry
for col in gene_cols:
    mean = data.groupby('Timestep')[col].mean()
    std  = data.groupby('Timestep')[col].std()
    plt.errorbar(timesteps, mean, yerr=std, label=col.replace('Average ', ''), fmt='-o')
plt.xlabel('Timestep')
plt.ylabel('Value')
plt.title('Time Series of Average Gene Values Across All Trials')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
timesteps  = data['Timestep'].unique()
# Auto-discovered from HDF5 columns — always in sync with METRIC_COLUMNS
death_cols = [c for c in data.columns if c.startswith('Deaths from ')]
for col in death_cols:
    mean = data.groupby('Timestep')[col].mean()
    std  = data.groupby('Timestep')[col].std()
    plt.errorbar(timesteps, mean, yerr=std, label=col, fmt='-o')
plt.xlabel('Timestep')
plt.ylabel('Mean cumulative deaths (+/- std across trials)')
plt.title('Agent Deaths Over Time - Mean Across Trials')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
timesteps       = data['Timestep'].unique()
average_species = data.groupby('Timestep')['Number of Species'].mean()
std_species     = data.groupby('Timestep')['Number of Species'].std()
plt.errorbar(timesteps, average_species, yerr=std_species, label='Average Number of Species', fmt='-o')
plt.xlabel('Timestep')
plt.ylabel('Number of Species')
plt.title('Number of Species Over Time Across All Trials')
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

finals['Number of Species'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Species count')
axes[0].set_title('Final Species Count per Trial')
axes[0].tick_params(axis='x', rotation=0)

finals['Population Count'].plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('Population')
axes[1].set_title('Final Population per Trial')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()